In [17]:
import time
import os

import porespy as ps
import numpy as np
import scipy as sc
import pypardiso

os.chdir("..")
%run .\pyflowsolver\volumeManager.py
%run .\pyflowsolver\sparseArray.py
%run .\pyflowsolver\fastLaplacian.py
%run .\pyflowsolver\darcySolver.py
os.chdir("notebooks")

In [25]:
SIZE = 30
vol = ps.generators.blobs(shape=(SIZE, SIZE, SIZE), blobiness=0.4, porosity=0.55)
vol, n_lab = sc.ndimage.label(vol)
vol = (vol==1)
if vol.sum() == 0:
    print('error')
else:
    print('image OK')

cond_vol = (vol==1)*100 #porosity map ndarray uint8 0..100
cond_vol = fast_laplacian_volume_generator(
    cond_vol, 
    (1., 1., 1.), 
    )

#cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.0000001
cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.00001

volume_manager = VolumeManager(cond_vol)

image OK


In [26]:
if vol.shape[0] <= 30: # 30 for a 64 Gb RAM system
    dense_A, dense_b = volume_manager.get_linear_system()
    solution_template = np.linalg.solve(dense_A, dense_b)
    raveled_template = volume_manager.ravel_dense_solution(solution_template)
else:
    raveled_template = None

In [20]:
solver = DarcySolver()
sparse_A, sparse_b = volume_manager.get_sparse_system_jit()

In [5]:
if sparse_b.size < 150000:
    start_time = time.perf_counter()
    solution, error, iterations = solver.solve_jit(
        sparse_A, 
        sparse_b, 
        parallel=12, 
        max_iterations=100000, 
        target_error=1e-6,
    )
    run_time = time.perf_counter() - start_time
    if raveled_template is not None:
        raveled_solution = volume_manager.ravel_sparse_solution(solution)
        diff = np.abs(raveled_solution - raveled_template)
        print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
    else:
        print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: [9.999151e-07]   Iterations: 4979   Mean Error: 0.03222523257136345   Max error: 0.22951099276542664   Run time: 4.447324499953538


In [6]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_cg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        sparse_b,
        max_iterations=max_iterations*100, # sqrt(n) for n x n system
        target_error=1.0e-9, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: 9.814411170614468e-10   Iterations: 76191   Mean Error: 2.3864673494244926e-05   Max error: 0.003132462501525879   Run time: 38.695833900012076


In [7]:
P_val, P_col_idx, P_row_ptr = _get_diagonal_preconditioner(
    A_val = sparse_A.val, 
    A_col_idx=sparse_A.col_idx, 
    A_row_ptr=sparse_A.row_ptr, 
    threads=1,
    )
print(P_val)
print(P_col_idx)
print(P_row_ptr)

start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_pcg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        P_val, 
        P_col_idx, 
        P_row_ptr,
        sparse_b,
        max_iterations=max_iterations*100, # sqrt(n) for n x n system
        target_error=1.0e-9, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

[-7.02396144e+02 -9.75158565e+02 -1.12012815e+03 ... -3.16812993e-01
 -1.01488595e-01 -4.28833536e-02]
[    0     1     2 ... 14582 14583 14584]
[    0     1     2 ... 14582 14583 14584]
Error: 9.678461254511732e-10   Iterations: 287   Mean Error: 1.2609434634214267e-06   Max error: 1.354515552520752e-05   Run time: 1.74366859998554


In [9]:
from pypardiso import spsolve
from scipy.sparse import csc_matrix, csr_matrix


In [22]:
A = csr_matrix( (sparse_A.val, sparse_A.col_idx, np.append(sparse_A.row_ptr,sparse_A.val.size)) )

In [23]:
spsolve(A, sparse_b)

array([0.73724332, 0.73091064, 0.72620097, ..., 0.18512747, 0.17630468,
       0.16572788])

In [27]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
A = csr_matrix( (sparse_A.val, sparse_A.col_idx, np.append(sparse_A.row_ptr,sparse_A.val.size)) )
solution = spsolve(A, sparse_b)
error = 0
iterations = 0
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: 0   Iterations: 0   Mean Error: 0.17996381223201752   Max error: 0.9992562532424927   Run time: 0.012248500250279903


In [28]:
solution

array([0.73724332, 0.73091064, 0.72620097, ..., 0.18512747, 0.17630468,
       0.16572788])